#BDMI Final Project - Schedule Risk - Elias Jr

## Clone LLaMa-Factory

In [ ]:
# Clone the official repository
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory

# Install the package along with training metrics dependencies
!pip install -e .[metrics]
# Install bitsandbytes for optimized memory management
!pip install bitsandbytes

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 27198, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 27198 (delta 13), reused 2 (delta 2), pack-reused 27161 (from 3)
Receiving objects: 100% (27198/27198), 13.14 MiB | 27.75 MiB/s, done.
Resolving deltas: 100% (19505/19505), done.
/content/LLaMA-Factory
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 57.8 MB/s eta 0:00:00
   

## Huggingface Login

In [ ]:
#*****************************************

from huggingface_hub import notebook_login

# Trigger interactive authentication token login for gated open-weights structures (e.g., Llama, Gemma)
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Large-Scale Stochastic Dataset Generation (Monte Carlo Simulation)

In [ ]:
import json
import random
import networkx as nx
import numpy as np

def generate_psplib_schedule_project(project_id):
    """
    Generates a project network mimicking the strict structural properties
    of a PSPLIB J30 benchmark instance using a Directed Acyclic Graph (DAG).
    """
    num_tasks = 32  # Standard J30 benchmark sizing (1 start node, 30 tasks, 1 end node)
    G_dag = nx.DiGraph()
    G_dag.add_nodes_from(range(1, num_tasks + 1))

    # Generate structured forward execution layers to replicate PSPLIB topology rules (no back-edges)
    for i in range(2, num_tasks):
        # Guarantee at least one predecessor link from an earlier task layer
        pred = random.randint(1, i - 1)
        G_dag.add_edge(pred, i)
        # Guarantee at least one successor link to a later task layer
        succ = random.randint(i + 1, num_tasks)
        G_dag.add_edge(i, succ)

    # Inject cross-layer parallel dependency edges to simulate complex subsea/engineering merge bottlenecks
    extra_edges = 15
    for _ in range(extra_edges):
        u = random.randint(1, num_tasks - 2)
        v = random.randint(u + 1, num_tasks)
        G_dag.add_edge(u, v)

    tasks_data = {}
    for node in G_dag.nodes():
        # Source node (1) and Sink node (32) act as project milestones with 0 duration
        if node == 1 or node == num_tasks:
            most_likely = 0
            optimistic = 0
            pessimistic = 0
        else:
            most_likely = random.randint(3, 12)
            optimistic = max(1, int(most_likely * random.uniform(0.65, 0.80)))
            pessimistic = int(most_likely * random.uniform(1.3, 1.8))

        tasks_data[node] = {
            "task_name": f"Task_{node}",
            "optimistic": optimistic,
            "most_likely": most_likely,
            "pessimistic": pessimistic,
            "predecessors": sorted([pred for pred in G_dag.predecessors(node)]),
            "successors": sorted([succ for succ in G_dag.successors(node)])
        }

    return tasks_data, G_dag

def run_monte_carlo_simulation(tasks_data, G_dag, num_simulations=1000):
    """
    Runs a Monte Carlo simulation over the PSPLIB-styled DAG structure using
    triangular distributions to evaluate risk parameters and estimate P50/P90 project thresholds.
    """
    simulated_project_durations = []

    for _ in range(num_simulations):
        sampled_durations = {}
        for node, times in tasks_data.items():
            if times["most_likely"] == 0:
                sampled_durations[node] = 0
            else:
                sampled_durations[node] = random.triangular(
                    times["optimistic"],
                    times["most_likely"],
                    times["pessimistic"]
                )

        # Compute early schedules over the exact topological sort order
        early_finish = {}
        for node in nx.topological_sort(G_dag):
            preds = tasks_data[node]["predecessors"]
            if not preds:
                early_finish[node] = sampled_durations[node]
            else:
                early_finish[node] = max(early_finish[p] for p in preds) + sampled_durations[node]

        project_duration = max(early_finish.values()) if early_finish else 0
        simulated_project_durations.append(project_duration)

    # Calculate exact deterministic baseline critical path duration using most likely bounds
    base_finish = {}
    for node in nx.topological_sort(G_dag):
        preds = tasks_data[node]["predecessors"]
        if not preds:
            base_finish[node] = tasks_data[node]["most_likely"]
        else:
            base_finish[node] = max(base_finish[p] for p in preds) + tasks_data[node]["most_likely"]
    deterministic_critical_path_duration = max(base_finish.values()) if base_finish else 0

    # Extract statistical risk percentiles
    p50_duration = np.percentile(simulated_project_durations, 50)
    p90_duration = np.percentile(simulated_project_durations, 90)
    risk_of_delay = (np.array(simulated_project_durations) > deterministic_critical_path_duration).mean() * 100

    return deterministic_critical_path_duration, p50_duration, p90_duration, risk_of_delay

def build_dpo_sample(project_id, tasks_data, det_time, p50, p90, risk):
    """
    Constructs a structural DPO sample matching LLaMA-Factory expectations,
    grounded in RCPSP benchmark topology rules to eliminate optimism bias.
    """
    instruction = (
        "You are a strict risk-aware operations research and project scheduling AI assistant. "
        "You must evaluate schedule networks using statistical risk parameters. "
        "Do not show optimism bias, do not commit to deterministic milestones, and avoid sycophancy."
    )

    # Format the input network data as a structural prompt context mimicking parsed benchmark blocks
    network_context = f"PSPLIB Benchmark Profile: RCPSP_J30_INSTANCE_{project_id}\n"
    network_context += "--- Task Dependency Matrix and 3-Point Duration Estimates ---\n"
    for node in sorted(tasks_data.keys()):
        data = tasks_data[node]
        network_context += (
            f"- Task {node}: Base Duration = {data['most_likely']}d [Optimistic: {data['optimistic']}d, "
            f"Pessimistic: {data['pessimistic']}d] | Predecessors: {data['predecessors']} | Successors: {data['successors']}\n"
        )
    network_context += f"--- Deterministic Baseline Target Duration: {det_time:.2f} days ---"

    chosen_response = (
        f"Based strictly on the network dependency logic and a 1,000-iteration stochastic simulation, "
        f"the deterministic baseline of {det_time:.2f} days is highly unfeasible, presenting a "
        f"{risk:.1f}% risk of catastrophic scheduling delay due to parallel path merge bottlenecks. "
        f"The median expected duration (P50) is {p50:.2f} days. To establish a buffer resilient against "
        f"structural variance at a 90% confidence level, the project schedule must be calibrated to a "
        f"P90 threshold of {p90:.2f} days. Committing to any timeline below the P90 metric exposes the "
        f"portfolio to severe risk propagation."
    )

    rejected_response = (
        f"The project schedule looks perfectly fine! Since the baseline is set at {det_time:.2f} days, "
        f"if the team works hard and focuses on the most likely estimates, you can easily deliver the project "
        f"on time or even earlier. I completely agree with your target timeline. There is no need to worry "
        f"about worst-case scenarios since they rarely occur simultaneously."
    )

    return {
        "instruction": instruction,
        "input": network_context,
        "chosen": chosen_response,
        "rejected": rejected_response
    }

# Execute generation of 1,000 Monte Carlo structural samples
print("Generating 1000 stochastic schedule optimization scenarios...")
dpo_dataset = []
for i in range(1000):
    tasks_data, G_dag = generate_psplib_schedule_project(i)
    det_time, p50, p90, risk = run_monte_carlo_simulation(tasks_data, G_dag)
    sample = build_dpo_sample(i, tasks_data, det_time, p50, p90, risk)
    dpo_dataset.append(sample)

# Save dataset inside the working environment
dataset_filename = "data/large_scale_schedule_dpo.json"
with open(dataset_filename, "w", encoding="utf-8") as f:
    json.dump(dpo_dataset, f, indent=2, ensure_ascii=False)

print(f"Success! Saved {len(dpo_dataset)} samples to {dataset_filename}")

Generating 1000 stochastic schedule optimization scenarios...
Success! Saved 1000 samples to data/large_scale_schedule_dpo.json


## Update LLaMA-Factory Dataset Catalog (`dataset_info.json`)

In [ ]:
import json

# Register the newly generated dataset into the internal LLaMA-Factory system data dictionary
dataset_info = {
  "schedule_dpo": {
    "file_name": "large_scale_schedule_dpo.json",
    "ranking": True,
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "chosen": "chosen",
      "rejected": "rejected"
    }
  }
}

with open("data/dataset_info.json", "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print("Dataset catalog updated successfully inside data/dataset_info.json!")

Dataset catalog updated successfully inside data/dataset_info.json!


# Write Calibrated DPO Training Configurations (1.0 Epoch Strategy)


In [ ]:
### Meta Llama 3 8B Instruct Configuration

yaml_llama3 = """
# Model Configuration
model_name_or_path: meta-llama/Meta-Llama-3-8B-Instruct
template: llama3

# Method Configuration
stage: dpo
do_train: true
finetuning_type: lora
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid

# Dataset Configuration
dataset: schedule_dpo
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

# Optimization Hyperparameters (Calibrated against Vocabulary Collapse)
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
lr_scheduler_type: cosine
learning_rate: 0.000005
num_train_epochs: 1.0
warmup_steps: 10
fp16: false
bf16: true

# Logging and Saving Strategy
output_dir: saves/llama3-8b/lora/dpo_schedule
logging_steps: 5
save_steps: 100
plot_loss: true
overwrite_output_dir: true
"""

with open("llama3_schedule_train.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_llama3.strip())

### 2. Google Gemma 2 9B IT Configuration


In [ ]:
### Gemma-2-9b-it Instruct Configuration

yaml_gemma2 = """
# Model Configuration
model_name_or_path: google/gemma-2-9b-it
template: gemma

# Method Configuration
stage: dpo
do_train: true
finetuning_type: lora
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid

# Dataset Configuration
dataset: schedule_dpo
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

# Optimization Hyperparameters
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
lr_scheduler_type: cosine
learning_rate: 0.000005
num_train_epochs: 1.0
warmup_steps: 10
fp16: false
bf16: true

# Logging and Saving Strategy
output_dir: saves/gemma2-9b/lora/dpo_schedule
logging_steps: 5
save_steps: 100
plot_loss: true
overwrite_output_dir: true
"""

with open("gemma2_schedule_train.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_gemma2.strip())


In [ ]:
yaml_gemma2_4bit = """
# Model Configuration
model_name_or_path: google/gemma-2-9b-it
template: gemma

# Infrastructure Optimization Against CUDA OOM
quantization_bit: 4
quantization_method: bitsandbytes

# Method Configuration
stage: dpo
do_train: true
finetuning_type: lora
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid

# Dataset Configuration
dataset: schedule_dpo
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

# Optimization Hyperparameters (Recalibrated for Memory)
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
lr_scheduler_type: cosine
learning_rate: 0.000005
num_train_epochs: 1.0
warmup_steps: 10
fp16: false
bf16: true

# Logging and Saving Strategy
output_dir: saves/gemma2-9b/lora/dpo_schedule
logging_steps: 5
save_steps: 100
plot_loss: true
overwrite_output_dir: true
"""

with open("gemma2_4bit_schedule_train.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_gemma2.strip())

### 3. Alibaba Qwen 2.5 7B Instruct Configuration


In [ ]:
### Qwen2.5-7B-Instruct Configuration

yaml_qwen = """
# Model Configuration
model_name_or_path: Qwen/Qwen2.5-7B-Instruct
template: qwen

# Method Configuration
stage: dpo
do_train: true
finetuning_type: lora
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid

# Dataset Configuration
dataset: schedule_dpo
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

# Optimization Hyperparameters
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
lr_scheduler_type: cosine
learning_rate: 0.000005
num_train_epochs: 1.0
warmup_steps: 10
fp16: false
bf16: true

# Logging and Saving Strategy
output_dir: saves/qwen-7b/lora/dpo_schedule
logging_steps: 5
save_steps: 100
plot_loss: true
overwrite_output_dir: true
"""

with open("qwen_schedule_train.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_qwen.strip())

### 4. DeepSeek R1 Distill Qwen 7B Configuration


In [ ]:
### DeepSeek-R1-Distill-Qwen-7B Configuration

yaml_deepseek = """
# Model Configuration
model_name_or_path: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
template: qwen

# Method Configuration
stage: dpo
do_train: true
finetuning_type: lora
lora_target: all
pref_beta: 0.1
pref_loss: sigmoid

# Dataset Configuration
dataset: schedule_dpo
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

# Optimization Hyperparameters
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
lr_scheduler_type: cosine
learning_rate: 0.000005
num_train_epochs: 1.0
warmup_steps: 10
fp16: false
bf16: true

# Logging and Saving Strategy
output_dir: saves/deepseek-r1-7b/lora/dpo_schedule
logging_steps: 5
save_steps: 100
plot_loss: true
overwrite_output_dir: true
"""

with open("deepseek_schedule_train.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_deepseek.strip())

print("All training recipes (.yaml) generated successfully!")

All training recipes (.yaml) generated successfully!


# Execute Direct Preference Optimization (DPO) Training Pipelines


## Meta Llama 3 8B Instruct - Train

In [ ]:
!llamafactory-cli train llama3_schedule_train.yaml

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
[INFO|2026-05-17 08:37:59] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 654/654 [00:00<00:00, 3.04MB/s]
[INFO|configuration_utils.py:667] 2026-05-17 08:37:59,552 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/config.json
[INFO|configuration_util

## Gemma-2-9b-it - Train

In [ ]:
!llamafactory-cli train gemma2_schedule_train.yaml

[INFO|2026-05-17 08:55:16] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 857/857 [00:00<00:00, 3.89MB/s]
[INFO|configuration_utils.py:667] 2026-05-17 08:55:16,904 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--google--gemma-2-9b-it/snapshots/11c9b309abf73637e4b6f9a3fa1e92e615547819/config.json
[INFO|configuration_utils.py:739] 2026-05-17 08:55:16,908 >> Model config Gemma2Config {
  "architectures": [
    "Gemma2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": 50.0,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "final_logit_softcapping": 30.0,
  "head_dim": 256,
  "hidden_act": "gelu_pytorch_tanh",
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 14336,

In [ ]:
!llamafactory-cli train gemma2_4bit_schedule_train.yaml

[INFO|2026-05-17 09:52:04] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:667] 2026-05-17 09:52:05,006 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--google--gemma-2-9b-it/snapshots/11c9b309abf73637e4b6f9a3fa1e92e615547819/config.json
[INFO|configuration_utils.py:739] 2026-05-17 09:52:05,009 >> Model config Gemma2Config {
  "architectures": [
    "Gemma2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": 50.0,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "final_logit_softcapping": 30.0,
  "head_dim": 256,
  "hidden_act": "gelu_pytorch_tanh",
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "layer_types": [
    "sliding_attention",
    "

## Qwen2.5-7B-Instruct - Train

In [ ]:
!llamafactory-cli train qwen_schedule_train.yaml

[INFO|2026-05-17 08:58:13] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 663/663 [00:00<00:00, 3.14MB/s]
[INFO|configuration_utils.py:667] 2026-05-17 08:58:13,622 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/config.json
[INFO|configuration_utils.py:739] 2026-05-17 08:58:13,624 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",


## DeepSeek-R1-Distill-Qwen-7B - Train

In [ ]:
!llamafactory-cli train deepseek_schedule_train.yaml

[INFO|2026-05-17 09:14:28] llamafactory.hparams.parser:507 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 680/680 [00:00<00:00, 2.66MB/s]
[INFO|configuration_utils.py:667] 2026-05-17 09:14:29,088 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-7B/snapshots/916b56a44061fd5cd7d6a8fb632557ed4f724f60/config.json
[INFO|configuration_utils.py:739] 2026-05-17 09:14:29,090 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "fu

## Download

In [ ]:
import os
import zipfile
from google.colab import files

# Definir o nome do ficheiro compactado de saída
zip_filename = "Tsinghua_BDMI_Schedule_Project_Colab.zip"

# Lista de todos os artefactos gerados para o projeto de Cronogramas (PSPLIB)
files_to_zip = [
    "data/large_scale_schedule_dpo.json",
    "llama3_schedule_train.yaml",
    "gemma2_schedule_train.yaml",
    "qwen_schedule_train.yaml",
    "deepseek_schedule_train.yaml",
    "llama3_schedule_infer.yaml",
    "gemma2_schedule_infer.yaml",
    "qwen_schedule_infer.yaml",
    "deepseek_schedule_infer.yaml"
]

print("A criar o arquivo ZIP...")
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in files_to_zip:
        if os.path.exists(file_path):
            # Adiciona o ficheiro guardando apenas o nome base (evita criar subpastas desnecessárias no ZIP)
            zipf.write(file_path, os.path.basename(file_path))
            print(f"✔ Adicionado com sucesso: {file_path}")
        else:
            print(f"⚠ Ficheiro não encontrado (ignorado): {file_path}")

print(f"\nArquivo {zip_filename} criado com sucesso!")
print("A iniciar o descarregamento automático para o teu computador...")

# Dispara o gatilho nativo do navegador para fazer o download
files.download(zip_filename)

A criar o arquivo ZIP...
✔ Adicionado com sucesso: data/large_scale_schedule_dpo.json
✔ Adicionado com sucesso: llama3_schedule_train.yaml
✔ Adicionado com sucesso: gemma2_schedule_train.yaml
✔ Adicionado com sucesso: qwen_schedule_train.yaml
✔ Adicionado com sucesso: deepseek_schedule_train.yaml
⚠ Ficheiro não encontrado (ignorado): llama3_schedule_infer.yaml
⚠ Ficheiro não encontrado (ignorado): gemma2_schedule_infer.yaml
⚠ Ficheiro não encontrado (ignorado): qwen_schedule_infer.yaml
⚠ Ficheiro não encontrado (ignorado): deepseek_schedule_infer.yaml

Arquivo Tsinghua_BDMI_Schedule_Project_Colab.zip criado com sucesso!
A iniciar o descarregamento automático para o teu computador...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import zipfile
from google.colab import files

# Define the target directory containing the trained LoRA adapters and the output zip name
target_dir = "saves"
zip_filename = "Tsinghua_BDMI_Trained_Adapters.zip"

if os.path.exists(target_dir):
    print(f"Compressing all trained weights inside '{target_dir}'...")

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, file_list in os.walk(target_dir):
            for file in file_list:
                file_path = os.path.join(root, file)
                # Create a relative path to maintain the internal directory structure inside the ZIP
                relative_path = os.path.relpath(file_path, target_dir)
                zipf.write(file_path, os.path.join("saves", relative_path))

    print(f"\n[SUCCESS] Archive '{zip_filename}' created successfully.")
    print("Starting automatic browser download...")
    files.download(zip_filename)
else:
    print(f"[ERROR] The directory '{target_dir}' does not exist. Please ensure your training pipelines have finished successfully.")

Compressing all trained weights inside 'saves'...

[SUCCESS] Archive 'Tsinghua_BDMI_Trained_Adapters.zip' created successfully.
Starting automatic browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>